# 25 - Probabilidad, probabilidad condicional y formula de Bayes en R

## Proposito de la sesion

Esta sesion es complementaria a la siguiente: aqui construimos la base conceptual que luego se usa en modelos probabilisticos.

En esta sesion vas a:

1. Repasar experimento aleatorio, espacio muestral y eventos.
2. Conectar probabilidad teorica con frecuencia empirica.
3. Entender union, interseccion, complemento e independencia.
4. Calcular probabilidad condicional con tablas y datos reales.
5. Incorporar la regla de la probabilidad total.
6. Entender y aplicar la formula de Bayes.
7. Cerrar con ejercicios que obliguen a interpretar, no solo a sustituir valores.


In [ ]:
# Configuracion minima para la sesion.
set.seed(2026)
options(scipen = 999)

# Titanic esta disponible en R base como tabla agregada.
titanic_tab <- as.data.frame(Titanic)

cat('Total de registros agregados:', nrow(titanic_tab), '\n')
cat('Total de personas representadas:', sum(titanic_tab$Freq), '\n\n')
head(titanic_tab)


## Ruta de la sesion

La progresion es intencional:

1. Empezamos con ejemplos pequenos donde se ve el espacio muestral completo.
2. Pasamos a frecuencias empiricas para conectar teoria con simulacion.
3. Introducimos independencia y probabilidad condicional.
4. Usamos `Titanic` para leer probabilidades dentro de datos reales.
5. Cerramos con Bayes y con un ejemplo donde la intuicion suele fallar.

La meta no es memorizar formulas, sino aprender a leer que significa cada condicion y cada denominador.


In [ ]:
# Vista rapida del numero de personas por clase y supervivencia.
tab_class_surv <- xtabs(Freq ~ Class + Survived, data = titanic_tab)
tab_sex_surv <- xtabs(Freq ~ Sex + Survived, data = titanic_tab)

tab_class_surv
tab_sex_surv


## 1) Experimento aleatorio, espacio muestral y eventos

Un **experimento aleatorio** es un proceso cuyo resultado no se conoce con certeza antes de realizarlo.

El **espacio muestral** contiene todos los resultados posibles.

Si lanzamos un dado justo, entonces:

$$S = \{1,2,3,4,5,6\}$$

Un **evento** es cualquier subconjunto de `S`.

Ejemplos:

- `A`: obtener un numero par.
- `B`: obtener un numero mayor que 4.

Si todos los resultados son equiprobables, entonces:

$$P(A) = \frac{|A|}{|S|}$$


In [ ]:
# Espacio muestral y eventos en un dado.
espacio <- 1:6
A <- c(2, 4, 6)   # numero par
B <- c(5, 6)      # numero mayor que 4

p_A <- length(A) / length(espacio)
p_B <- length(B) / length(espacio)

data.frame(
  evento = c('A_par', 'B_mayor_que_4'),
  probabilidad = c(p_A, p_B)
)


## 2) Union, interseccion y complemento

Con eventos podemos combinar informacion:

- `A ? B`: ocurre `A` o ocurre `B`.
- `A ? B`: ocurren ambos.
- `A^c`: no ocurre `A`.

Las reglas mas utiles son:

$$P(A^c) = 1 - P(A)$$

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

La resta evita contar dos veces la parte compartida.


In [ ]:
# Operaciones entre eventos.
union_AB <- union(A, B)
inter_AB <- intersect(A, B)
comp_A <- setdiff(espacio, A)

p_union_directa <- length(union_AB) / length(espacio)
p_inter <- length(inter_AB) / length(espacio)
p_union_formula <- p_A + p_B - p_inter
p_comp_A <- length(comp_A) / length(espacio)

list(
  union = union_AB,
  interseccion = inter_AB,
  complemento_A = comp_A,
  prob_union_directa = p_union_directa,
  prob_union_formula = p_union_formula,
  prob_complemento_A = p_comp_A
)


## 3) Probabilidad teorica vs frecuencia empirica

Muchas veces conocemos la probabilidad por simetria, como en un dado justo.
Otras veces solo podemos estimarla observando datos o simulaciones.

La idea central es que, al repetir muchas veces el experimento, la frecuencia relativa suele estabilizarse cerca de la probabilidad real.

Eso no significa que cada tramo pequeno sea estable. Significa que la trayectoria acumulada tiende a hacerse menos volatil.


In [ ]:
# Aproximacion empirica de P(numero par).
set.seed(2026)

n <- 600
corrida <- sample(espacio, size = n, replace = TRUE)
proporcion_par <- cumsum(corrida %in% A) / seq_len(n)

plot(
  proporcion_par,
  type = 'l',
  lwd = 2,
  col = 'steelblue',
  ylim = c(0, 1),
  xlab = 'numero de lanzamientos',
  ylab = 'proporcion acumulada',
  main = 'Frecuencia empirica de obtener un numero par'
)
abline(h = p_A, col = 'firebrick', lty = 2, lwd = 2)
legend(
  'topright',
  legend = c('empirica', 'teorica'),
  col = c('steelblue', 'firebrick'),
  lty = c(1, 2),
  lwd = 2,
  bty = 'n'
)


## 4) Independencia

Dos eventos son **independientes** si conocer uno no cambia la probabilidad del otro.

Una forma de verificarlo es:

$$P(A \cap B) = P(A)P(B)$$

La independencia no significa que los eventos sean "distintos" o "parecidos".
Significa algo mas preciso: que la ocurrencia de uno no aporta informacion sobre el otro.


In [ ]:
# Contraste entre un caso independiente y uno dependiente.
set.seed(2026)

# Caso 1: dos monedas justas independientes.
moneda_1 <- sample(c('cara', 'cruz'), size = 20000, replace = TRUE)
moneda_2 <- sample(c('cara', 'cruz'), size = 20000, replace = TRUE)

A_ind <- moneda_1 == 'cara'
B_ind <- moneda_2 == 'cara'

# Caso 2: en un dado, "par" y "mayor que 3" no son independientes.
A_dep <- espacio %in% c(2, 4, 6)
B_dep <- espacio %in% c(4, 5, 6)

comparacion <- data.frame(
  escenario = c('monedas_independientes', 'eventos_en_dado_dependientes'),
  P_interseccion = c(mean(A_ind & B_ind), mean(A_dep & B_dep)),
  P_producto = c(mean(A_ind) * mean(B_ind), mean(A_dep) * mean(B_dep))
)

comparacion


## 5) Probabilidad condicional

La **probabilidad condicional** responde preguntas como:

> "Si ya se que ocurrio `B`, que tan probable es `A`?"

Formalmente:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

siempre que `P(B) > 0`.

La idea clave es que cambia el universo de referencia.
Ya no miramos todos los casos, sino solo aquellos donde `B` ocurrio.


In [ ]:
# Probabilidad condicional en Titanic.
survival_by_sex <- prop.table(tab_sex_surv, margin = 1)
survival_by_class <- prop.table(tab_class_surv, margin = 1)

round(survival_by_sex, 3)
round(survival_by_class, 3)

cat('P(Survived = Yes | Sex = Female):', round(survival_by_sex['Female', 'Yes'], 4), '\n')
cat('P(Survived = Yes | Class = 1st):', round(survival_by_class['1st', 'Yes'], 4), '\n')


## 6) El orden de la condicion importa

Estas dos expresiones no responden la misma pregunta:

- `P(Survived = Yes | Sex = Female)`
- `P(Sex = Female | Survived = Yes)`

La primera fija el sexo y pregunta por supervivencia.
La segunda fija la supervivencia y pregunta por sexo.

Cambiar la condicion cambia el denominador. Si cambia el denominador, cambia la interpretacion.


In [ ]:
# Comparacion de dos direcciones de condicionamiento.
p_yes_given_female <- prop.table(tab_sex_surv, margin = 1)['Female', 'Yes']
p_female_given_yes <- prop.table(xtabs(Freq ~ Survived + Sex, data = titanic_tab), margin = 1)['Yes', 'Female']

cat('P(Survived = Yes | Sex = Female):', round(p_yes_given_female, 4), '\n')
cat('P(Sex = Female | Survived = Yes):', round(p_female_given_yes, 4), '\n\n')

par(mfrow = c(1, 2))
barplot(
  survival_by_sex[, 'Yes'],
  col = c('tomato', 'steelblue'),
  ylim = c(0, 1),
  main = 'P(sobrevive | sexo)',
  ylab = 'probabilidad'
)
barplot(
  prop.table(xtabs(Freq ~ Survived + Sex, data = titanic_tab), margin = 1)['Yes', ],
  col = c('tomato', 'steelblue'),
  ylim = c(0, 1),
  main = 'P(sexo | sobrevive)',
  ylab = 'probabilidad'
)
par(mfrow = c(1, 1))


## 7) Regla de la probabilidad total

A veces un evento puede ocurrir a traves de varios caminos que forman una particion del espacio.
En ese caso sumamos probabilidades ponderadas.

Si `A_1, A_2, ..., A_k` forman una particion, entonces:

$$P(B) = \sum_{i=1}^{k} P(B \mid A_i)P(A_i)$$

Esta regla es importante porque la formula de Bayes la usa en el denominador.


In [ ]:
# Verificacion de la probabilidad total con la variable Class.
p_class <- prop.table(rowSums(tab_class_surv))
p_yes_given_class <- prop.table(tab_class_surv, margin = 1)[, 'Yes']
p_yes_total_directa <- sum(tab_class_surv[, 'Yes']) / sum(tab_class_surv)
p_yes_total_por_clase <- sum(p_yes_given_class * p_class)

data.frame(
  metodo = c('directo', 'probabilidad_total'),
  valor = round(c(p_yes_total_directa, p_yes_total_por_clase), 6)
)


## 8) Formula de Bayes

La formula de Bayes permite invertir un condicionamiento.

$$P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}$$

Lectura practica:

- `P(A)` es la probabilidad previa o **prior**.
- `P(B | A)` mide que tan compatible es la evidencia con la hipotesis.
- `P(A | B)` es la probabilidad actualizada despues de observar la evidencia.

Bayes no crea informacion de la nada: reorganiza informacion que ya estaba en probabilidades condicionales y marginales.


In [ ]:
# Verificacion de Bayes con Titanic.
p_female <- sum(tab_sex_surv['Female', ]) / sum(tab_sex_surv)
p_yes <- sum(tab_sex_surv[, 'Yes']) / sum(tab_sex_surv)
p_yes_given_female <- tab_sex_surv['Female', 'Yes'] / sum(tab_sex_surv['Female', ])

p_female_given_yes_directa <- tab_sex_surv['Female', 'Yes'] / sum(tab_sex_surv[, 'Yes'])
p_female_given_yes_bayes <- p_yes_given_female * p_female / p_yes

data.frame(
  metodo = c('directo', 'bayes'),
  probabilidad = round(c(p_female_given_yes_directa, p_female_given_yes_bayes), 6)
)


## 9) Ejemplo donde Bayes corrige una intuicion ingenua

Una confusion muy comun es pensar:

> "Si una prueba es muy buena y sale positiva, entonces casi seguro tengo la condicion".

Eso puede ser falso si la condicion es muy rara.

Para entenderlo, construyamos un escenario simple de test diagnostico.


In [ ]:
# Escenario sintetico de test diagnostico.
prevalencia <- 0.01     # 1% de la poblacion tiene la condicion
sensibilidad <- 0.95    # P(test positivo | condicion)
especificidad <- 0.90   # P(test negativo | no condicion)

p_condicion <- prevalencia
p_no_condicion <- 1 - prevalencia
p_pos_given_cond <- sensibilidad
p_pos_given_no_cond <- 1 - especificidad

p_positivo <- p_pos_given_cond * p_condicion + p_pos_given_no_cond * p_no_condicion
p_condicion_given_positivo <- p_pos_given_cond * p_condicion / p_positivo

# Llevarlo a conteos sobre una poblacion de 100000 personas.
poblacion <- 100000
conteos <- data.frame(
  grupo = c('condicion_y_test_positivo', 'condicion_y_test_negativo', 'sin_condicion_y_test_positivo', 'sin_condicion_y_test_negativo'),
  personas = c(
    poblacion * p_condicion * p_pos_given_cond,
    poblacion * p_condicion * (1 - p_pos_given_cond),
    poblacion * p_no_condicion * p_pos_given_no_cond,
    poblacion * p_no_condicion * especificidad
  )
)
conteos$personas <- round(conteos$personas, 0)

conteos
cat('\nP(test positivo):', round(p_positivo, 4), '\n')
cat('P(condicion | test positivo):', round(p_condicion_given_positivo, 4), '\n')


## 10) La prevalencia cambia el resultado de Bayes

Con la misma sensibilidad y especificidad, la probabilidad posterior puede cambiar mucho si cambia la prevalencia.

Eso explica por que una prueba puede ser util en una poblacion de alto riesgo y mucho menos convincente en una poblacion general.


In [ ]:
# Como cambia P(condicion | positivo) cuando cambia la prevalencia.
prevalencias <- seq(0.001, 0.5, length.out = 200)
posterior_pos <- (sensibilidad * prevalencias) /
  (sensibilidad * prevalencias + (1 - especificidad) * (1 - prevalencias))

plot(
  prevalencias,
  posterior_pos,
  type = 'l',
  lwd = 2,
  col = 'darkgreen',
  xlab = 'prevalencia',
  ylab = 'P(condicion | positivo)',
  main = 'Impacto de la prevalencia en Bayes'
)
abline(v = prevalencia, col = 'gray40', lty = 2)
abline(h = p_condicion_given_positivo, col = 'firebrick', lty = 2)


## 11) Ideas clave y errores comunes

Ideas clave:

1. Una probabilidad condicional siempre redefine el conjunto de referencia.
2. La regla de la probabilidad total conecta probabilidades condicionadas con probabilidades globales.
3. Bayes sirve para actualizar creencias con evidencia.
4. Una evidencia aparentemente fuerte puede no ser concluyente si el evento base es raro.

Errores comunes:

1. Confundir `P(A|B)` con `P(B|A)`.
2. Ignorar el denominador y leer solo el numerador.
3. Hablar de independencia sin verificar que significa exactamente.
4. Concluir demasiado a partir de una prueba positiva sin considerar prevalencia.


In [ ]:
# Checklist minimo de autoevaluacion.
checklist <- c(
  'Puedo distinguir espacio muestral, evento y probabilidad.',
  'Puedo explicar que cambia cuando condiciono en un evento.',
  'Puedo usar la probabilidad total para recomponer una probabilidad global.',
  'Puedo interpretar la formula de Bayes en palabras, no solo en simbolos.',
  'Puedo explicar por que la prevalencia afecta la interpretacion de un resultado positivo.'
)

checklist


## 12) Ejercicios para pensar (no copiar codigo)

Primero responde en texto. Luego, si hace falta, usa R para validar tus ideas.

1. Construye dos eventos en un mismo espacio muestral que tengan interseccion no vacia pero que sean independientes. Explica por que eso no es una contradiccion.
2. Explica con tus palabras por que `P(A|B)` y `P(B|A)` suelen confundirse y dise?a una regla mental para no mezclarlas.
3. En `Titanic`, compara `P(Survived = Yes | Class = 1st)` con la probabilidad global de supervivencia. Que te dice esa diferencia sobre la clase como fuente de informacion.
4. Imagina una enfermedad rara y una prueba con 99% de sensibilidad y 99% de especificidad. Explica por que aun asi pueden aparecer muchos falsos positivos en una campana masiva.
5. Dise?a un ejemplo donde la regla de la probabilidad total sea indispensable para resolver el problema. Justifica por que no basta con mirar una sola probabilidad condicional.
6. Describe una situacion real en la que asumir independencia seria claramente incorrecto. Explica como ese error podria sesgar una conclusion.
7. Si una probabilidad posterior cambia mucho cuando actualizas con una evidencia, que te dice eso sobre la combinacion entre prior y evidencia observada.


In [ ]:
# Espacio para resolver ejercicios.
# Primero escribe argumentos en lenguaje natural.
# Luego valida con tablas, simulaciones o calculos si hace falta.
